# End-to-End SFT Pipeline: PEFT + QLoRA on Hugging Face

**Supervised Fine-Tuning (SFT)** of an open LLM with 4-bit quantization and LoRA adapters,
built on Transformers + Datasets + PEFT + TRL + bitsandbytes.

---

### What this notebook does

```text
Training Dataset  ->  Cleaning / Formatting  ->  Chat Template  ->  Tokenizer
      -> Pretrained Base LLM -> 4-bit Quantization -> LoRA Adapters
      -> Supervised Fine-Tuning -> Save Adapter -> Reload Base + Adapter
      -> Inference -> Compare Base vs Fine-Tuned
```

### The four ideas in one paragraph

| Term | Meaning here |
| --- | --- |
| **SFT** | We show the model pairs of `input -> desired answer` and nudge its weights so it becomes more likely to produce answers like the ones we showed it. |
| **PEFT** | Parameter-Efficient Fine-Tuning: instead of updating all ~0.5B weights, we train a small set of *extra* parameters. |
| **LoRA** | The concrete PEFT method used here: small trainable low-rank matrices `A` and `B` are injected next to selected frozen weight matrices. Only `A` and `B` receive gradients. |
| **QLoRA** | LoRA on top of a base model whose frozen weights are stored in **4-bit**. The memory saving comes from the quantized frozen base; the learning happens in the (small, higher-precision) LoRA adapters. |

### How to run

1. **Runtime -> Change runtime type -> GPU.** An **L4** or **A100** is recommended (both support bfloat16). A T4 works but is slower and falls back to fp16.
2. **Runtime -> Run all.**
3. Expected wall-clock on an L4: roughly **8-15 minutes** end to end, most of it in the training cell.

> **Note on outputs:** this notebook is distributed with **cleared outputs**. Every number, loss curve and
> model response you see after running it is produced live on your machine. Nothing below is a pasted or
> illustrative result.


## 1. Environment Setup

### 1.1 Install pinned packages

Versions are pinned to a mutually compatible, current set (verified against PyPI metadata) so the
notebook is reproducible. **PyTorch is deliberately not pinned** - we use whatever CUDA build Colab
already ships, which avoids a multi-GB re-download and CUDA/driver mismatches.

These are *current* APIs (Transformers 5.x, TRL 1.x, PEFT 0.20). They differ in several places from
older Llama-2 QLoRA tutorials - notably `dtype=` replaces `torch_dtype=`, `warmup_steps` replaces
`warmup_ratio`, and `SFTConfig` uses `max_length` rather than `max_seq_length`.

The last three pins (`jinja2`, `matplotlib`, `pandas`) already exist on Colab and arrive
transitively anyway. They are listed explicitly because the Hugging Face `huggingface-llm-trainer`
skill's *Reliability Principle 3* ("Create Atomic, Self-Contained Scripts") warns against relying
on that: its worked example is a script that broke silently after someone dropped a dependency that
"didn't look necessary". `jinja2` is what renders the chat template - without it
`apply_chat_template()` raises, and every formatting cell below depends on it.

In [ ]:
# Run this FIRST, before importing anything else.
# In a fresh Colab runtime nothing heavy is imported yet, so these upgrades take effect
# without needing a runtime restart.

%pip install -q \
    "transformers==5.15.0" \
    "trl==1.10.0" \
    "peft==0.20.0" \
    "datasets==5.0.1" \
    "accelerate==1.14.0" \
    "bitsandbytes==0.50.1" \
    "jinja2>=3.1" \
    "matplotlib" \
    "pandas"

print("Install step finished.")
print("If any cell below fails with an ImportError, use Runtime -> Restart session and Run all again.")

### 1.2 Environment report + hard GPU requirement

QLoRA needs CUDA: `bitsandbytes` 4-bit kernels are GPU-only. This cell **stops the notebook** with a
clear message if no GPU is attached, rather than failing later with a cryptic error.

In [ ]:
import sys
import platform

import torch

print("=" * 68)
print("ENVIRONMENT REPORT")
print("=" * 68)
print(f"Python           : {platform.python_version()} ({sys.executable})")
print(f"PyTorch          : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n\n"
        "  NO CUDA GPU DETECTED.\n"
        "  This notebook requires a GPU because QLoRA uses bitsandbytes 4-bit CUDA kernels.\n"
        "  Fix: Runtime -> Change runtime type -> Hardware accelerator -> GPU (L4 or A100), then Run all.\n"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"CUDA version     : {torch.version.cuda}")
print(f"GPU              : {gpu_name}")
print(f"Compute capability: sm_{gpu_capability[0]}{gpu_capability[1]}")
print(f"GPU memory       : {gpu_total_gb:.1f} GB")

# Pick the mixed-precision mode. We call the exact same predicate that
# transformers' TrainingArguments validates against, so `bf16=True` below can
# never be rejected later with "Your setup doesn't support bf16/gpu".
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()

if SUPPORTS_BF16:
    COMPUTE_DTYPE = torch.bfloat16
    USE_BF16, USE_FP16 = True, False
    print("Mixed precision  : bfloat16 (preferred - no loss scaling needed)")
else:
    COMPUTE_DTYPE = torch.float16
    USE_BF16, USE_FP16 = False, True
    print("Mixed precision  : float16 (bfloat16 unavailable on this GPU)")

# Native bf16 arithmetic needs Ampere (sm_80) or newer. Colab's L4 (sm_89) and
# A100 (sm_80) qualify; the older T4 (sm_75) does not.
if gpu_capability[0] < 8:
    print()
    print("  WARNING: this GPU predates Ampere (e.g. a T4). Everything below still runs,")
    print("           but training will be noticeably slower. An L4 or A100 is recommended.")

print("=" * 68)

import transformers, datasets, peft, trl, bitsandbytes, accelerate
print(f"transformers {transformers.__version__} | datasets {datasets.__version__} | "
      f"peft {peft.__version__} | trl {trl.__version__} | "
      f"bitsandbytes {bitsandbytes.__version__} | accelerate {accelerate.__version__}")

---

## 2. Configuration

Everything tunable lives in this one cell so the rest of the notebook reads top-to-bottom without
hidden magic numbers.

In [ ]:
# ---------------------------------------------------------------- MODEL
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# ---------------------------------------------------------------- DATASET
DATASET_NAME = "b-mc2/sql-create-context"

# Deliberately small so the whole notebook finishes in minutes on Colab.
# Raise these for a better model; the pipeline is unchanged.
TRAIN_SIZE = 2000
EVAL_SIZE  = 200
SEED       = 42

# ---------------------------------------------------------------- QLoRA / LoRA
LORA_R             = 16          # rank of the low-rank update
LORA_ALPHA         = 32          # scaling; effective scale is alpha / r = 2.0
LORA_DROPOUT       = 0.05
LORA_TARGET_MODULES = [          # which frozen matrices get an adapter
    "q_proj", "k_proj", "v_proj", "o_proj",     # attention projections
    "gate_proj", "up_proj", "down_proj",        # MLP projections
]
LORA_BIAS      = "none"
LORA_TASK_TYPE = "CAUSAL_LM"

# ---------------------------------------------------------------- TRAINING
OUTPUT_DIR        = "./sft_output"     # trainer checkpoints / logs
ADAPTER_DIR       = "./sft_adapter"    # final LoRA adapter (the deliverable artifact)
NUM_TRAIN_EPOCHS  = 2
PER_DEVICE_BS     = 8
GRAD_ACCUM_STEPS  = 2                  # effective batch = 8 * 2 = 16
LEARNING_RATE     = 2e-4               # LoRA tolerates a much higher LR than full fine-tuning
MAX_SEQ_LENGTH    = 256               # measured: the longest formatted example here is 215 tokens
LOGGING_STEPS     = 5
EVAL_STEPS        = 25

# ---------------------------------------------------------------- INFERENCE
MAX_NEW_TOKENS = 128

SYSTEM_PROMPT = "You are a helpful assistant that answers questions about a SQL database."

print(f"Model   : {MODEL_NAME}")
print(f"Dataset : {DATASET_NAME}")
print(f"Adapter : {ADAPTER_DIR}")

### Why this base model?

**`Qwen/Qwen2.5-0.5B-Instruct`** was chosen over Llama 2 (the model in the older reference notebook) for
three practical reasons. It is **fully open and ungated** - no license acceptance, no HF token, so
*Run All* works on a fresh Colab account. It is **small enough to be honest about**: at 0.5B parameters
the 4-bit base occupies well under 1 GB, so training finishes in minutes on any Colab Pro GPU instead
of timing out. And it ships an **official ChatML chat template** plus standard Llama-style projection
names (`q_proj`, `v_proj`, ...), so both `tokenizer.apply_chat_template()` and PEFT's `target_modules`
work without inventing special tokens by hand.

> Swapping in a bigger model is a one-line change: set `MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"`.
> Nothing else in the notebook needs to change.

### 2.1 Pre-flight verification

The Hugging Face `huggingface-llm-trainer` skill opens its reliability guidance with
*Principle 1: Always Verify Before Use* - "Never assume repos, datasets, or resources exist. Verify
with tools first." Its stated cost/benefit is 5-10 seconds of checking against hours of failed GPU
time, and the failures it prevents are dull ones: a renamed repo, a typo, a moved dataset.

So before anything touches the GPU, confirm both resources actually resolve. The dataset check uses
the Datasets Server `/is-valid` endpoint, which is the validation entry point documented by the
`huggingface-datasets` skill.

In [ ]:
import json
import urllib.error
import urllib.parse
import urllib.request

def _get_json(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.load(response)

print("Pre-flight checks (skill Reliability Principle 1: verify before use)")
print("-" * 68)

# --- 1. Does the base model exist and is it ungated? -------------------------
try:
    model_info = _get_json(f"https://huggingface.co/api/models/{MODEL_NAME}")
    gated = model_info.get("gated", False)
    print(f"  [OK]   model   {MODEL_NAME}")
    print(f"         gated={gated}  downloads={model_info.get('downloads', 0):,}")
    if gated:
        raise RuntimeError(
            f"{MODEL_NAME} is gated. You would need to accept its license and supply an "
            f"HF token before this notebook can run unattended."
        )
except urllib.error.HTTPError as exc:
    raise RuntimeError(
        f"Base model '{MODEL_NAME}' could not be resolved on the Hugging Face Hub "
        f"(HTTP {exc.code}). Check the spelling in the configuration cell above."
    ) from exc

# --- 2. Is the dataset present and served by the Datasets Server? ------------
# /is-valid is the validation endpoint documented by the huggingface-datasets skill.
try:
    quoted = urllib.parse.quote(DATASET_NAME, safe="")
    validity = _get_json(f"https://datasets-server.huggingface.co/is-valid?dataset={quoted}")
    print(f"  [OK]   dataset {DATASET_NAME}")
    print(f"         viewer={validity.get('viewer')}  preview={validity.get('preview')}")
except urllib.error.HTTPError as exc:
    raise RuntimeError(
        f"Dataset '{DATASET_NAME}' could not be validated via the Datasets Server "
        f"(HTTP {exc.code}). Check the spelling in the configuration cell above."
    ) from exc

print("-" * 68)
print("Both resources resolved. Safe to spend GPU time.")

---

## 3. Dataset

We use **`b-mc2/sql-create-context`** (~78k examples, CC-BY-4.0): natural-language questions paired
with the SQL that answers them, given a table schema.

Each raw record has three fields, which map onto the three parts of a supervised training example:

| Raw field | Role in SFT | Meaning |
| --- | --- | --- |
| `question` + `context` | **input / instruction** | What we give the model: the schema and the question. |
| `answer` | **desired output** | What we want the model to produce: the SQL query, and nothing else. |
| the pair of the two | **training example** | One `(input, desired output)` pair. The loss is computed on the desired output only. |

This task is a good SFT demonstration because the *format* of a correct answer is unambiguous - bare
SQL - which makes the before/after difference easy to see rather than a matter of taste.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_NAME, split="train")

print(f"Loaded {len(raw_dataset):,} raw examples")
print(f"Columns: {raw_dataset.column_names}")
print()

# --- show several RAW examples, before any formatting -------------------------
for i in range(3):
    ex = raw_dataset[i]
    print("-" * 68)
    print(f"RAW EXAMPLE {i}")
    print("-" * 68)
    print(f"  context  (schema)   : {ex['context']}")
    print(f"  question (instruction): {ex['question']}")
    print(f"  answer   (desired output): {ex['answer']}")
    print()

### Cleaning and splitting

A validity filter, then a train/eval split. We keep the subset small on purpose - this is a
demonstration of the *pipeline*, not an attempt to reach state-of-the-art text-to-SQL.

> **On the filter:** this particular dataset is already clean, so you should expect **0 examples
> dropped**. The filter is a guard, not busywork - it is what keeps the pipeline correct if you point
> `DATASET_NAME` at a messier source. Cleaning that silently does nothing is worth *seeing* rather
> than assuming, which is why the count is printed.

In [ ]:
# --- clean: drop records with an empty field, and over-long schemas ----------
def is_valid(example):
    fields_present = all(
        isinstance(example[k], str) and example[k].strip()
        for k in ("question", "context", "answer")
    )
    # Guard against schemas so long they would be truncated by MAX_SEQ_LENGTH.
    # (In this dataset the longest schema is 489 characters, so nothing is dropped.)
    schema_reasonable = len(example["context"]) < 600
    return fields_present and schema_reasonable


clean_dataset = raw_dataset.filter(is_valid)
print(f"After cleaning : {len(clean_dataset):,} examples "
      f"({len(raw_dataset) - len(clean_dataset):,} dropped)")

# --- subsample, then split into train / eval --------------------------------
subset = clean_dataset.shuffle(seed=SEED).select(range(TRAIN_SIZE + EVAL_SIZE))
split = subset.train_test_split(test_size=EVAL_SIZE, seed=SEED)

train_raw = split["train"]
eval_raw = split["test"]

print(f"Train examples : {len(train_raw):,}")
print(f"Eval  examples : {len(eval_raw):,}")

assert len(train_raw) > 0 and len(eval_raw) > 0, "Dataset split produced an empty set"

---

## 4. Chat / Instruction Formatting

The model was instruction-tuned with a specific **chat template**. Rather than inventing special
tokens by hand (`[INST]`, `<<SYS>>`, ... as older Llama-2 notebooks do), we build plain
`{"role": ..., "content": ...}` message dicts and let the tokenizer's *official* template render them.

We emit TRL's **conversational prompt-completion** format:

```python
{"prompt":     [{"role": "system", ...}, {"role": "user", ...}],
 "completion": [{"role": "assistant", "content": "<the SQL>"}]}
```

Using `prompt`/`completion` rather than a single `messages` list matters: TRL detects this shape and
automatically turns on **`completion_only_loss`**, so the loss is computed on the SQL answer only and
*not* on the schema and question we fed in. That is what makes this supervised fine-tuning of an
answer rather than plain language modelling over the whole string.

In [ ]:
def to_prompt_completion(example):
    # The "input" side: schema + question, as a single user turn.
    user_content = (
        "Database schema:\n"
        f"{example['context']}\n\n"
        f"Question: {example['question']}"
    )
    # The "desired output" side: the bare SQL query.
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "completion": [
            {"role": "assistant", "content": example["answer"]},
        ],
    }


ORIGINAL_COLUMNS = train_raw.column_names

train_dataset = train_raw.map(to_prompt_completion, remove_columns=ORIGINAL_COLUMNS)
eval_dataset = eval_raw.map(to_prompt_completion, remove_columns=ORIGINAL_COLUMNS)

print(f"Formatted columns: {train_dataset.column_names}")
print()
import json
print("One formatted example (message dicts, before the template is applied):")
print(json.dumps(train_dataset[0], indent=2)[:1200])

### 4.1 Tokenizer + what the model actually receives

Now we load the tokenizer and render the same example through the model's chat template. This is the
exact string that gets tokenized during training - special tokens and all.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Qwen ships a distinct pad token; make sure it is set so batching works.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer      : {tokenizer.__class__.__name__}")
print(f"Vocab size     : {len(tokenizer):,}")
print(f"EOS token      : {tokenizer.eos_token!r} (id {tokenizer.eos_token_id})")
print(f"PAD token      : {tokenizer.pad_token!r} (id {tokenizer.pad_token_id})")
print()

example = train_dataset[0]

# The prompt half, with the generation prompt appended (what inference sees).
rendered_prompt = tokenizer.apply_chat_template(
    example["prompt"], tokenize=False, add_generation_prompt=True
)
# The full training string: prompt + the assistant turn we want the model to learn.
rendered_full = tokenizer.apply_chat_template(
    example["prompt"] + example["completion"], tokenize=False
)

print("=" * 68)
print("FULLY FORMATTED TRAINING EXAMPLE (exact string fed to the tokenizer)")
print("=" * 68)
print(rendered_full)
print("=" * 68)
print()

n_full = len(tokenizer(rendered_full)["input_ids"])
n_prompt = len(tokenizer(rendered_prompt)["input_ids"])
print(f"Total tokens        : {n_full}")
print(f"Prompt tokens       : {n_prompt}   <- loss is MASKED here")
print(f"Completion tokens   : {n_full - n_prompt}   <- loss is computed here")

### 4.2 Validate the formatted dataset against TRL's contract

The `huggingface-llm-trainer` skill is blunt about this: *"50%+ of training failures are due to
dataset format issues"*, and it tells you to validate any dataset that is not already a known
TRL-native one before spending GPU time.

Running the skill's own `dataset_inspector.py` against `b-mc2/sql-create-context` reports
**`[SFT] NEEDS MAPPING`** - the raw columns are `question` / `context` / `answer`, which TRL cannot
consume directly. That is exactly the mapping section 4 just performed.

The inspector's *suggested* mapping flattens everything into a single `text` field. We deliberately
map to conversational `prompt` / `completion` instead, which the skill's `training_methods.md` lists
as a supported SFT format. The reason is loss masking: a flat `text` field trains the model on the
schema and the question as well as the answer, whereas `prompt`/`completion` lets TRL mask the prompt
and train only on the SQL. Same mapping requirement, stricter target format.

Rather than trusting that by eye, the check below calls **TRL's own `is_conversational`** - the
predicate `SFTTrainer` itself uses internally to decide how to tokenize.

In [ ]:
from trl import is_conversational

sample = train_dataset[0]

# The exact predicate SFTTrainer uses to route tokenization.
assert is_conversational(sample), (
    "TRL does not recognise this dataset as conversational. Expected 'prompt' and 'completion' "
    "to each be a list of {'role': ..., 'content': ...} dicts."
)

# prompt+completion is the key set that switches on completion-only loss masking.
key_set = {k for k in sample.keys() if k in ("prompt", "completion", "messages", "chosen", "rejected", "label")}
assert key_set == {"prompt", "completion"}, f"Unexpected TRL key set: {key_set}"

# Roles must be ordered so the chat template renders a real assistant turn last.
assert sample["prompt"][-1]["role"] == "user", "The prompt must end on a user turn"
assert sample["completion"][0]["role"] == "assistant", "The completion must be an assistant turn"

# The prompt render MUST be a strict prefix of the prompt+completion render, or the
# prompt/completion token boundary - and therefore the loss mask - would be wrong.
prompt_only = tokenizer.apply_chat_template(
    sample["prompt"], tokenize=False, add_generation_prompt=True
)
prompt_plus_completion = tokenizer.apply_chat_template(
    sample["prompt"] + sample["completion"], tokenize=False
)
assert prompt_plus_completion.startswith(prompt_only), (
    "The rendered prompt is not a prefix of the rendered prompt+completion. "
    "completion_only_loss would mask the wrong tokens."
)

print("Dataset format validation")
print("-" * 68)
print(f"  [OK] TRL is_conversational()      : True")
print(f"  [OK] key set                      : {sorted(key_set)}  -> completion_only_loss enabled")
print(f"  [OK] prompt ends on role          : {sample['prompt'][-1]['role']}")
print(f"  [OK] completion role              : {sample['completion'][0]['role']}")
print(f"  [OK] prompt is a strict prefix    : loss mask boundary is well defined")
print("-" * 68)
print(f"Validated {len(train_dataset):,} train / {len(eval_dataset):,} eval examples in this format.")

---

## 5. Base Model + 4-bit Quantization (the "Q" in QLoRA)

`BitsAndBytesConfig` tells `from_pretrained` to store the frozen base weights in **4-bit NF4** instead
of 16-bit. Two details matter:

- **`bnb_4bit_quant_type="nf4"`** - NormalFloat4, the quantization grid introduced in the QLoRA paper.
  It is information-theoretically better suited to normally-distributed weights than plain fp4.
- **`bnb_4bit_compute_dtype`** - weights are *stored* in 4 bits but *dequantized on the fly* to this
  dtype for each matmul. Storage is 4-bit; arithmetic is not.

`bnb_4bit_use_double_quant=True` additionally quantizes the quantization constants themselves, saving
a further ~0.4 bits per parameter.

We load the model **once** here and use it for both the baseline generations and, in a moment, as the
frozen trunk under the LoRA adapters. Running the "before" prompts through the *same* 4-bit model that
training will use means the before/after difference is attributable to the adapter alone, not to a
change in quantization.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # store base weights in 4 bits
    bnb_4bit_quant_type="nf4",                # NF4 grid (QLoRA paper)
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,     # dequantize to bf16/fp16 for the matmul
    bnb_4bit_use_double_quant=True,           # quantize the quantization constants too
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},        # pin the whole model to GPU 0 (single-GPU training)
    dtype=COMPUTE_DTYPE,       # NOTE: transformers 5.x renamed `torch_dtype` -> `dtype`
)
# Note: the trainer will set `use_cache=False` itself once gradient checkpointing is on.
# We leave it enabled here so the baseline generations below run at full speed.

print(f"Loaded: {MODEL_NAME}")
print(f"  is_loaded_in_4bit : {getattr(model, 'is_loaded_in_4bit', False)}")
print(f"  device            : {next(model.parameters()).device}")
print(f"  GPU memory used   : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print()
print(model.config.__class__.__name__,
      f"| hidden={model.config.hidden_size} layers={model.config.num_hidden_layers}")

---

## 6. Baseline Inference (BEFORE fine-tuning)

We pick evaluation prompts from the **held-out eval split** - the model never sees these during
training - and record what the un-adapted base model produces. These strings are stored and reused
verbatim in the final comparison table, so the "before" column is a genuine recording rather than a
re-run under different conditions.

Decoding is **greedy** (`do_sample=False`) on both sides of the comparison, so the difference we see
later is caused by the adapter and not by sampling randomness.

In [ ]:
EVAL_PROMPTS = [eval_raw[i] for i in range(4)]   # 4 held-out examples

print("Evaluation prompts (held out from training):")
for i, ex in enumerate(EVAL_PROMPTS):
    print(f"\n[{i}] schema  : {ex['context']}")
    print(f"    question: {ex['question']}")
    print(f"    reference SQL: {ex['answer']}")

In [ ]:
@torch.no_grad()
def generate_sql(a_model, question: str, schema: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    # Build the SAME prompt shape used in training, then render it with the official chat template
    # and append the generation prompt so the model knows it is the assistant's turn.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Database schema:\n{schema}\n\nQuestion: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(a_model.device)

    was_training = a_model.training
    a_model.eval()
    output_ids = a_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,                       # greedy: deterministic, fair comparison
        pad_token_id=tokenizer.pad_token_id,
    )
    if was_training:
        a_model.train()

    # Keep only the newly generated tokens, dropping the echoed prompt.
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("Running BASELINE inference on the 4-bit base model (no adapter)...\n")

baseline_responses = []
for i, ex in enumerate(EVAL_PROMPTS):
    response = generate_sql(model, ex["question"], ex["context"])
    baseline_responses.append(response)
    print("=" * 68)
    print(f"PROMPT {i}: {ex['question']}")
    print("-" * 68)
    print("BASE MODEL OUTPUT:")
    print(response)
    print()

assert len(baseline_responses) == len(EVAL_PROMPTS)
print(f"Stored {len(baseline_responses)} baseline responses for the final comparison.")

---

## 7. PEFT / LoRA Configuration

LoRA freezes the pretrained weight `W` and learns a low-rank correction beside it:

```text
        frozen (4-bit)              trainable (small, higher precision)
   h  =  W x                +      (alpha / r) * B @ (A @ x)

   W : d_out x d_in         A : r x d_in      B : d_out x r        r << min(d_in, d_out)
```

Only `A` and `B` get gradients. Because `r = 16` while the projection dimensions are in the hundreds
or thousands, the trainable parameter count comes out around **1% of the model**.

Parameter notes:

- **`r = 16`** - rank of the update. Higher rank = more capacity and more parameters.
- **`lora_alpha = 32`** - the update is scaled by `alpha / r = 2.0`. Alpha decouples the effective
  learning-rate scale from the choice of rank.
- **`lora_dropout = 0.05`** - dropout on the LoRA branch; a small amount of regularization for a
  small dataset.
- **`target_modules`** - we adapt *all* attention and MLP projections. Adapting only `q_proj`/`v_proj`
  is cheaper; adapting everything generally learns new formats faster, which is what we want here.
- **`bias = "none"`** - do not train bias terms. Standard for LoRA; keeps the adapter tiny.
- **`task_type = "CAUSAL_LM"`** - tells PEFT this is next-token prediction, so it wires the adapter
  into the right forward signature.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias=LORA_BIAS,
    task_type=LORA_TASK_TYPE,
)

print(peft_config)

---

## 8. Supervised Fine-Tuning with TRL

`SFTTrainer` takes the quantized model, attaches the LoRA adapters from `peft_config`, tokenizes the
prompt/completion dataset with the chat template, and runs a standard training loop.

Settings are deliberately conservative - the goal is a run that **finishes reliably**, not a
leaderboard score:

| Setting | Value | Why |
| --- | --- | --- |
| epochs | 2 | Enough passes over 2,000 examples for the output format to stick. |
| batch / accumulation | 8 x 2 | Effective batch 16; small enough for any Colab GPU. |
| learning rate | 2e-4 | LoRA tolerates ~100x the LR of full fine-tuning - only the adapters move. |
| scheduler | cosine, 10% warmup | Warmup avoids a destabilizing first few steps. |
| `max_length` | 256 | Measured: the longest formatted example in this subset is 215 tokens, so nothing is truncated. (TRL calls this `max_length`, **not** `max_seq_length`.) |
| mixed precision | bf16 or fp16 | Chosen from the GPU capability detected in section 1.2. |
| `report_to` | `"none"` | No W&B / Trackio prompt interrupting *Run All*. |

> **Deliberate deviation from the skill.** The `huggingface-llm-trainer` skill says to *"always
> include Trackio"* for real-time monitoring, and for its target environment - Hugging Face Jobs,
> where training runs detached on a remote GPU and logs are the only window into it - that is
> plainly right. Here the trainer prints its metrics directly into the cell output below, and
> enabling Trackio would require an HF token and a Trackio Space, which breaks the tokenless
> *Run all* this notebook is built around. Section 9 recovers the same information from
> `trainer.state.log_history`. To opt in anyway, add `report_to="trackio"` plus
> `project=` and `run_name=` to the config above.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # --- schedule -----------------------------------------------------------
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,          # transformers 5.x: a float < 1 means "ratio of total steps"
    max_grad_norm=1.0,

    # --- sequences ----------------------------------------------------------
    max_length=MAX_SEQ_LENGTH,   # TRL uses `max_length` (NOT `max_seq_length`)
    packing=False,
    # The `trl-training` skill lists `--packing` under "Slow Training" as the fix for short
    # sequences, and ours are short (median 91 tokens, max 215). It stays off anyway: in TRL
    # 1.x the default 'bfd' packing strategy forces padding-free mode, which is only reliable
    # with a FlashAttention backend and warns otherwise. The `huggingface-llm-trainer` skill's
    # Reliability Principle 2 ("Prioritize Reliability Over Performance") settles the tie -
    # this run is already only a few minutes long, so there is no speed problem to solve.
    # completion_only_loss is left unset: TRL turns it on automatically for
    # prompt/completion datasets, so the loss ignores the prompt tokens.

    # --- precision / speed --------------------------------------------------
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=False,   # plenty of VRAM for a 0.5B 4-bit model; ~20% faster off

    # --- logging / eval / checkpoints ---------------------------------------
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    eval_strategy="steps",       # requires eval_dataset below, or training would hang
    eval_steps=EVAL_STEPS,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",

    seed=SEED,
)

trainer = SFTTrainer(
    model=model,                     # the already-loaded 4-bit base model
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,       # MUST be provided when eval_strategy != "no"
    processing_class=tokenizer,      # transformers 5.x: `processing_class`, not `tokenizer`
    peft_config=peft_config,         # LoRA adapters are attached here
)

print("Trainer ready.")

### 8.1 How much of the model are we actually training?

In [ ]:
trainable_params, total_params = trainer.model.get_nb_trainable_parameters()
trainable_pct = 100.0 * trainable_params / total_params

print("=" * 68)
print("PARAMETER BUDGET")
print("=" * 68)
print(f"Total parameters      : {total_params:>14,}")
print(f"Trainable parameters  : {trainable_params:>14,}")
print(f"Percentage trainable  : {trainable_pct:>13.4f} %")
print(f"Frozen parameters     : {total_params - trainable_params:>14,}")
print("=" * 68)
print()
print("Only the LoRA A/B matrices receive gradients. Everything else stays frozen")
print("in its 4-bit representation - that is the whole point of QLoRA.")
print()

# Show a couple of the actual trainable tensors, to make it concrete.
print("Sample of trainable tensors:")
shown = 0
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        print(f"  {name}  ->  {tuple(param.shape)}  ({param.dtype})")
        shown += 1
        if shown >= 4:
            break

assert trainable_params > 0, "No trainable parameters - the LoRA adapters were not attached"

### 8.2 Train

Live training output follows. Loss values, step counts and runtime below are produced by this run.

In [ ]:
import time

print(f"Starting SFT: {len(train_dataset):,} train / {len(eval_dataset):,} eval examples")
print(f"Effective batch size: {PER_DEVICE_BS} x {GRAD_ACCUM_STEPS} = {PER_DEVICE_BS * GRAD_ACCUM_STEPS}")
print()

train_start = time.time()
train_result = trainer.train()
train_wall_seconds = time.time() - train_start

print()
print("=" * 68)
print("TRAINING COMPLETE")
print("=" * 68)
for key, value in train_result.metrics.items():
    print(f"  {key:28s}: {value}")
print(f"  {'wall_clock_seconds':28s}: {train_wall_seconds:.1f}")
print(f"  {'peak_gpu_memory_gb':28s}: {torch.cuda.max_memory_allocated() / 1024**3:.2f}")

---

## 9. Training Metrics

`trainer.state.log_history` holds every logged event: training-loss entries every `logging_steps`
steps, and evaluation entries every `eval_steps` steps.

In [ ]:
import pandas as pd

log_history = trainer.state.log_history

train_logs = [r for r in log_history if "loss" in r and "eval_loss" not in r]
eval_logs = [r for r in log_history if "eval_loss" in r]

# Capture these now: the trainer object is torn down in section 11.
global_steps_done = trainer.state.global_step
epochs_done = trainer.state.epoch
final_train_loss = train_logs[-1]["loss"] if train_logs else float("nan")
final_eval_loss = eval_logs[-1]["eval_loss"] if eval_logs else float("nan")

print(f"Training-loss records : {len(train_logs)}")
print(f"Evaluation records    : {len(eval_logs)}")
print(f"Global steps completed: {global_steps_done}")
print(f"Epochs completed      : {epochs_done:.2f}")
print(f"Wall-clock runtime    : {train_wall_seconds:.1f} s "
      f"({train_wall_seconds / 60:.1f} min)")
print()

if train_logs:
    print("First / last training loss:")
    print(f"  step {train_logs[0]['step']:>4}  loss {train_logs[0]['loss']:.4f}")
    print(f"  step {train_logs[-1]['step']:>4}  loss {train_logs[-1]['loss']:.4f}")

if eval_logs:
    print()
    print("Evaluation loss:")
    display(pd.DataFrame(eval_logs)[["step", "epoch", "eval_loss"]].round(4))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))

if train_logs:
    ax.plot([r["step"] for r in train_logs],
            [r["loss"] for r in train_logs],
            marker="o", markersize=3, linewidth=1.4, label="training loss")

if eval_logs:
    ax.plot([r["step"] for r in eval_logs],
            [r["eval_loss"] for r in eval_logs],
            marker="s", markersize=6, linewidth=1.6, linestyle="--", label="eval loss")

ax.set_xlabel("training step")
ax.set_ylabel("loss")
ax.set_title(f"SFT loss - {MODEL_NAME} + LoRA r={LORA_R}")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

---

## 10. Save the Fine-Tuned Adapter

`trainer.save_model()` on a PEFT model writes **only the adapter**, not a copy of the base LLM.

Why the artifact is so small: the frozen 4-bit trunk never changed, so there is nothing to save about
it - it can always be re-downloaded from the Hub by name. The only thing training produced is the set
of LoRA `A`/`B` matrices, which is roughly 1% of the parameter count and stored in a normal float
dtype. So instead of shipping another multi-hundred-megabyte model, we ship a few megabytes that
"snap onto" the public base model. That is also why you can keep many task-specific adapters around
for the price of one base model.

In [ ]:
import os

trainer.save_model(ADAPTER_DIR)      # saves adapter weights + config (+ tokenizer files)

print(f"Adapter saved to: {os.path.abspath(ADAPTER_DIR)}")
print()
print("Generated files:")
total_bytes = 0
for filename in sorted(os.listdir(ADAPTER_DIR)):
    path = os.path.join(ADAPTER_DIR, filename)
    size = os.path.getsize(path)
    total_bytes += size
    print(f"  {filename:32s} {size / 1024:10.1f} KB")

print(f"\n  {'TOTAL':32s} {total_bytes / 1024**2:10.2f} MB")
print()
print("adapter_config.json records the base model id, rank, alpha and target modules,")
print("which is how PEFT knows what to reattach the weights to at load time.")

### 10.1 Optional: persist the adapter off this machine

Key takeaway #4 of the `huggingface-llm-trainer` skill is that the training environment is
**ephemeral** - *"without push, all results lost"*. The skill says that about Hugging Face Jobs, but
a Colab runtime is ephemeral in exactly the same way: when it disconnects or is recycled,
`./sft_adapter` goes with it.

This cell is **opt-in and off by default**, so *Run all* still works on a fresh account with no
token. Flip a flag if you want to keep the adapter.

In [ ]:
# Set ONE of these to persist the adapter beyond this runtime.
PUSH_TO_HUB = False          # requires a write token; pushes a few MB, not the base model
DOWNLOAD_LOCALLY = False     # zips the adapter and triggers a browser download

HUB_MODEL_ID = "your-username/qwen2.5-0.5b-sql-lora"   # only used when PUSH_TO_HUB is True

if PUSH_TO_HUB:
    from huggingface_hub import create_repo, login, upload_folder
    login()                                   # prompts for a token with write access
    # Upload straight from the saved directory: the artifact on disk IS the deliverable,
    # so this does not depend on any model object still being in memory.
    create_repo(HUB_MODEL_ID, repo_type="model", exist_ok=True)
    upload_folder(repo_id=HUB_MODEL_ID, folder_path=ADAPTER_DIR, repo_type="model")
    print(f"Adapter pushed to https://huggingface.co/{HUB_MODEL_ID}")

elif DOWNLOAD_LOCALLY:
    import shutil
    archive = shutil.make_archive("sft_adapter", "zip", ADAPTER_DIR)
    print(f"Created {archive} ({os.path.getsize(archive) / 1024**2:.2f} MB)")
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print("Not running in Colab - the archive is on disk at the path above.")

else:
    print("Skipped: the adapter lives only in this runtime.")
    print(f"  {os.path.abspath(ADAPTER_DIR)}")
    print()
    print("That is fine for a demonstration - the notebook reloads it from that path in the")
    print("next section. But it disappears when this runtime is recycled. Set PUSH_TO_HUB or")
    print("DOWNLOAD_LOCALLY above if you want to keep it.")

---

## 11. Reload: Base Model + Saved Adapter

To prove the saved artifact is genuinely reusable, we **discard the in-memory trainer and model
entirely**, free the GPU, then rebuild the fine-tuned model from scratch:

```text
    Pretrained Base LLM (re-quantized to 4-bit)
                 +
    ./sft_adapter  (the files we just wrote)
                 v
        Fine-Tuned Model
```

Nothing from the training session is carried over except the files on disk.

In [ ]:
import gc

# --- tear down everything from training -------------------------------------
# Guarded so this cell stays safe to re-run.
for _name in ("trainer", "model"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"GPU memory after teardown: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print()

# --- rebuild from disk -------------------------------------------------------
from peft import PeftModel

base_model_reloaded = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=COMPUTE_DTYPE,
)

finetuned_model = PeftModel.from_pretrained(base_model_reloaded, ADAPTER_DIR)
finetuned_model.eval()

n_adapter_modules = sum(1 for n, _ in finetuned_model.named_modules() if "lora_A" in n)

print("Reloaded fine-tuned model from disk.")
print(f"  type                : {type(finetuned_model).__name__}")
print(f"  active adapter      : {finetuned_model.active_adapters}")
print(f"  LoRA modules found  : {n_adapter_modules}")
print(f"  base is 4-bit       : {getattr(base_model_reloaded, 'is_loaded_in_4bit', False)}")
print(f"  GPU memory          : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

assert n_adapter_modules > 0, "Reloaded model has no LoRA modules attached"

---

## 12. Post-Training Inference and Base vs Fine-Tuned Comparison

The **same** four held-out prompts, the **same** greedy decoding, the **same** 4-bit base weights.
The only difference between the two columns is the LoRA adapter we just trained and reloaded.

In [ ]:
print("Running inference on the reloaded FINE-TUNED model...\n")

finetuned_responses = []
for i, ex in enumerate(EVAL_PROMPTS):
    response = generate_sql(finetuned_model, ex["question"], ex["context"])
    finetuned_responses.append(response)
    print("=" * 68)
    print(f"PROMPT {i}: {ex['question']}")
    print("-" * 68)
    print("FINE-TUNED OUTPUT:")
    print(response)
    print()

assert len(finetuned_responses) == len(EVAL_PROMPTS)

In [ ]:
# --- side-by-side, plain text ------------------------------------------------
for i, ex in enumerate(EVAL_PROMPTS):
    print("#" * 78)
    print(f"# PROMPT {i}")
    print("#" * 78)
    print(f"Schema   : {ex['context']}")
    print(f"Question : {ex['question']}")
    print()
    print("--- BEFORE SFT (base model) " + "-" * 44)
    print(baseline_responses[i])
    print()
    print("--- AFTER SFT (base + LoRA adapter) " + "-" * 36)
    print(finetuned_responses[i])
    print()
    print("--- REFERENCE (dataset ground truth) " + "-" * 35)
    print(ex["answer"])
    print()

In [ ]:
# --- rendered comparison table ----------------------------------------------
from html import escape
from IPython.display import HTML, display

rows = []
for i, ex in enumerate(EVAL_PROMPTS):
    rows.append(
        "<tr>"
        f"<td style='vertical-align:top;padding:8px;border:1px solid #999;width:26%'>"
        f"<b>Prompt {i}</b><br><br>{escape(ex['question'])}"
        f"<br><br><code style='font-size:11px;color:#666'>{escape(ex['context'][:160])}</code></td>"
        f"<td style='vertical-align:top;padding:8px;border:1px solid #999;width:37%;white-space:pre-wrap;font-family:monospace;font-size:12px'>{escape(baseline_responses[i])}</td>"
        f"<td style='vertical-align:top;padding:8px;border:1px solid #999;width:37%;white-space:pre-wrap;font-family:monospace;font-size:12px'>{escape(finetuned_responses[i])}</td>"
        "</tr>"
    )

table_html = (
    "<table style='border-collapse:collapse;width:100%;font-size:13px'>"
    "<thead><tr>"
    "<th style='padding:8px;border:1px solid #999;text-align:left'>Prompt</th>"
    "<th style='padding:8px;border:1px solid #999;text-align:left'>Base Model (BEFORE SFT)</th>"
    "<th style='padding:8px;border:1px solid #999;text-align:left'>Fine-Tuned Model (AFTER SFT)</th>"
    "</tr></thead><tbody>" + "".join(rows) + "</tbody></table>"
)

display(HTML(table_html))

### Reading the comparison

> The two columns above are **live output from this run** - not a transcript pasted into the notebook.
> What follows is how to read them, not a claim about what they say.

What SFT moves is **behaviour**, not knowledge. The base model already "knows" SQL. What it has not
been told is that in *this* task the expected response is a bare query, so it falls back on general
chat-assistant habits - which typically means a preamble, a fenced code block, and an explanation
afterwards. The **Base Model** column is where you see whatever habits it actually fell back on.

The **Fine-Tuned** column is the same weights plus an adapter trained on `(schema + question -> SQL)`
pairs whose answers are all bare queries. If the training worked, the output distribution has shifted
toward the shape of those answers: a single SQL statement, emitted directly, terminated by the chat
template's end-of-turn token.

That is the whole claim SFT makes - *show the model examples of `input -> desired answer`, and it
becomes more likely to produce responses like those examples*.

**What this run does not demonstrate:** query *correctness* on hard joins and aggregations. That is a
separate axis needing far more data and compute than a few hundred steps. Compare against the
reference SQL printed in the previous cell and judge the two things separately - a fine-tuned answer
can be perfectly formatted and still wrong.

---

## 13. Sanity Checks

Every claim this notebook makes, re-verified programmatically. A failure here raises with a message
that names what broke.

In [ ]:
checks = []

def check(label, condition, detail=""):
    checks.append((label, bool(condition), detail))

check("CUDA / GPU was used",
      torch.cuda.is_available() and torch.cuda.max_memory_allocated() > 0,
      f"GPU: {gpu_name}")

check("Dataset is non-empty",
      len(train_dataset) > 0 and len(eval_dataset) > 0,
      f"{len(train_dataset)} train / {len(eval_dataset)} eval")

check("Dataset is in prompt/completion chat format",
      "prompt" in train_dataset.column_names and "completion" in train_dataset.column_names,
      f"columns={train_dataset.column_names}")

check("Base model loaded in 4-bit",
      getattr(base_model_reloaded, "is_loaded_in_4bit", False),
      f"{MODEL_NAME}")

check("PEFT adapters attached",
      n_adapter_modules > 0,
      f"{n_adapter_modules} LoRA modules")

check("Trainable parameter count > 0",
      trainable_params > 0,
      f"{trainable_params:,} ({trainable_pct:.4f}%)")

check("Training completed",
      global_steps_done > 0 and train_result.metrics.get("train_runtime", 0) > 0,
      f"{global_steps_done} steps in {train_wall_seconds:.1f}s")

check("Training loss decreased",
      len(train_logs) >= 2 and train_logs[-1]["loss"] < train_logs[0]["loss"],
      f"{train_logs[0]['loss']:.4f} -> {train_logs[-1]['loss']:.4f}" if len(train_logs) >= 2 else "n/a")

check("Adapter directory exists",
      os.path.isdir(ADAPTER_DIR) and len(os.listdir(ADAPTER_DIR)) > 0,
      os.path.abspath(ADAPTER_DIR))

check("Adapter weights file present",
      any(f.startswith("adapter_model") for f in os.listdir(ADAPTER_DIR)),
      ", ".join(f for f in os.listdir(ADAPTER_DIR) if f.startswith("adapter")))

check("Adapter reloaded from disk",
      isinstance(finetuned_model, PeftModel),
      type(finetuned_model).__name__)

check("Inference works after reload",
      len(finetuned_responses) == len(EVAL_PROMPTS)
      and all(isinstance(r, str) and r.strip() for r in finetuned_responses),
      f"{len(finetuned_responses)} non-empty responses")

check("Base and fine-tuned outputs differ",
      baseline_responses != finetuned_responses,
      "the adapter changed the model's behaviour")

print("=" * 74)
print("SANITY CHECKS")
print("=" * 74)
failures = []
for label, passed, detail in checks:
    mark = "PASS" if passed else "FAIL"
    print(f"  [{mark}] {label:38s} {detail}")
    if not passed:
        failures.append(label)
print("=" * 74)

if failures:
    raise AssertionError(
        "The following sanity checks FAILED:\n  - " + "\n  - ".join(failures)
        + "\n\nScroll up to the corresponding section to see what went wrong."
    )
print(f"All {len(checks)} checks passed.")

---

## 14. Final Architecture

```text
                    SFT TRAINING PIPELINE

 Dataset
    |
    v
 Instruction / Chat Formatting
    |
    v
 Tokenizer
    |
    v
 +----------------------------+
 |     Pretrained Base LLM    |
 |       4-bit weights        |
 |                            |
 |      mostly FROZEN         |
 +-------------+--------------+
               |
          LoRA Adapters
           TRAINABLE
               |
               v
          SFT Trainer
               |
               v
         Trained Adapter
               |
               v
       Save adapter files


                    INFERENCE

       Pretrained Base LLM
               +
        Trained Adapter
               |
               v
        Fine-Tuned Model
               |
               v
            Response
```

### Where each box lives in this notebook

| Box | Section |
| --- | --- |
| Dataset | 3 - `load_dataset`, clean, split |
| Instruction / Chat Formatting | 4 - `to_prompt_completion` + `apply_chat_template` |
| Tokenizer | 4.1 - `AutoTokenizer` |
| Pretrained Base LLM, 4-bit, frozen | 5 - `BitsAndBytesConfig` + `from_pretrained` |
| LoRA Adapters, trainable | 7 - `LoraConfig` |
| SFT Trainer | 8 - `SFTTrainer` / `SFTConfig` |
| Save adapter files | 10 - `trainer.save_model("./sft_adapter")` |
| Base LLM + Trained Adapter | 11 - `PeftModel.from_pretrained` |
| Response | 12 - `generate_sql` |

---

## 15. Final Summary

In [ ]:
adapter_mb = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)
) / 1024**2

print("=" * 74)
print("SFT PIPELINE COMPLETE")
print("=" * 74)
for line in [
    "Dataset loaded",
    "Dataset formatted",
    "Base model loaded",
    "Baseline inference completed",
    "4-bit QLoRA configured",
    "LoRA adapters attached",
    "SFT training completed",
    "Adapter saved",
    "Adapter reloaded",
    "Post-training inference completed",
    "Base vs fine-tuned responses compared",
]:
    print(f"  [OK] {line}")

print()
print("=" * 74)
print("RUN REPORT")
print("=" * 74)
print(f"  Base model            : {MODEL_NAME}")
print(f"  Dataset               : {DATASET_NAME}")
print(f"  Training examples     : {len(train_dataset):,}")
print(f"  Evaluation examples   : {len(eval_dataset):,}")
print(f"  Training steps        : {global_steps_done}")
print(f"  Epochs completed      : {epochs_done:.2f}")
print(f"  Trainable parameters  : {trainable_params:,}")
print(f"  Total parameters      : {total_params:,}")
print(f"  Trainable %           : {trainable_pct:.4f} %")
print(f"  Final training loss   : {final_train_loss:.4f}")
print(f"  Final eval loss       : {final_eval_loss:.4f}")
print(f"  Training runtime      : {train_wall_seconds:.1f} s ({train_wall_seconds/60:.1f} min)")
print(f"  GPU                   : {gpu_name} ({gpu_total_gb:.1f} GB)")
print(f"  Mixed precision       : {'bf16' if USE_BF16 else 'fp16'}")
print(f"  Adapter location      : {os.path.abspath(ADAPTER_DIR)}")
print(f"  Adapter size on disk  : {adapter_mb:.2f} MB")
print("=" * 74)

---

### Reusing the adapter elsewhere

The `./sft_adapter` directory is the entire deliverable of this training run. Anywhere you have the
base model available, three lines rebuild the fine-tuned model:

```python
from transformers import AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
model = PeftModel.from_pretrained(base, "./sft_adapter")
```

Two useful follow-ups, both out of scope here:

- **`model.merge_and_unload()`** folds the LoRA weights into the base weights, producing a standalone
  model with no PEFT dependency at inference time. Note that merging into a *4-bit* base is lossy -
  merge into the fp16 base instead if you need this.
- **`model.push_to_hub("your-username/qwen2.5-0.5b-sql-lora")`** publishes the few-MB adapter to the
  Hub, where anyone can attach it to the public base model.
